# Kubeflow Notebooks as the Gateway to Kubernetes

### An end-to-end ML workflow: distributed data processing → TPU training → inference

This notebook is the **single control plane** for a realistic ML workflow. From one
JupyterLab Workspace running on a small CPU pod, we drive the full power of a
Kubernetes cluster:

| Stage | What runs | Where | K8s primitive |
|-------|-----------|-------|---------------|
| **1. Data processing** | Apache Spark ETL over Fashion-MNIST | 1 driver + 4 executor pods | Kubeflow SDK `SparkClient` |
| **2. Model training** | Data-parallel JAX training | Multi-host Cloud TPU slice | Kubeflow Trainer `TrainJob` (JobSet) |
| **3. Inference** | Model server | CPU Deployment (2 replicas) | `Deployment` + `Service` |

All three stages share a **GCS bucket** (mounted via GCSFuse), so data and model
artifacts flow between them without any object-storage plumbing.

> **The punchline:** the notebook pod itself is *tiny* (0.1 CPU). It never runs
> the heavy work — it **orchestrates** it. This is Kubeflow Notebooks acting as
> the developer's gateway to the whole cluster.

We also demonstrate the feature ML engineers actually feel the pain of every
day: **Pause & Resume** (snapshot/restore) of the notebook itself — walk away
from a long-running orchestration, snapshot the pod, and restore it later with
kernel state, variables, and running cells intact.


## Architecture

```
                    Kubeflow Notebook (this pod, 0.1 CPU)
                    "the gateway to Kubernetes"
                              │
          ┌───────────────────┼───────────────────────┐
          │ submit            │ submit                 │ deploy
          ▼                   ▼                        ▼
  ┌───────────────┐   ┌────────────────┐      ┌──────────────────┐
  │ Stage 1: ETL  │   │ Stage 2: Train │      │ Stage 3: Serve   │
  │ Spark:        │   │ 2 TPU hosts    │      │ 2 CPU replicas   │
  │ 1 drv + 4 exec│   │ (JobSet)       │      │ (Deployment)     │
  │(Spark SDK)    │   │                │      │                  │
  └──────┬────────┘   └───────┬────────┘      └────────┬─────────┘
         │ write               │ read / write           │ read
         ▼                     ▼                        ▼
   ┌─────────────────────────────────────────────────────────┐
   │        Shared GCS Bucket (mounted via GCSFuse)          │
   │  raw/  processed/{train,test}/  model/{params,metrics}   │
   └─────────────────────────────────────────────────────────┘
```

Everything below runs from this notebook. Let's go.


## 0. Setup & environment check

We confirm the SDKs are available and that this notebook can talk to the
Kubernetes API using its ServiceAccount. The `demo/jobs` package is on the path
so we can import the pipeline helpers.


In [ ]:
import os
import sys
import subprocess

# This notebook is copied to /home/jovyan/demo by scripts/run_demo.sh, alongside
# the `jobs` package. Make that directory importable regardless of the CWD.
DEMO_DIR = os.getcwd()
if not os.path.isdir(os.path.join(DEMO_DIR, "jobs")):
    DEMO_DIR = "/home/jovyan/demo"
if DEMO_DIR not in sys.path:
    sys.path.insert(0, DEMO_DIR)

import kubeflow.trainer  # noqa: F401
print("kubeflow.trainer import OK; DEMO_DIR =", DEMO_DIR)

# Verify the notebook's ServiceAccount can create the resources we need.
for res in ["trainjobs.trainer.kubeflow.org", "sparkapplications.sparkoperator.k8s.io"]:
    out = subprocess.run(["kubectl", "auth", "can-i", "create", res],
                         capture_output=True, text=True)
    print(f"can-i create {res.split('.')[0]:15s}:", out.stdout.strip() or out.stderr.strip())


### Provision shared storage (GCS)

We use a **GCS bucket** as the shared data bus between all three stages. GKE
mounts this bucket automatically via **GCSFuse**, so it looks like a regular
local volume (`/data`) to our code, but with the scalability and persistence
of object storage.


In [ ]:
bucket_name = os.environ.get("DEMO_BUCKET", "sizhang-gke-dev-ml-demo-data")
print(f"Using GCS bucket: {bucket_name}")

# Ensure the bucket exists (idempotent).
# Note: In a real environment, you'd ensure the ServiceAccount has IAM permissions.
# subprocess.run(["gcloud", "storage", "buckets", "create", f"gs://{bucket_name}", "--location=us-west1"], capture_output=True)
# !gcloud storage buckets list gs://{bucket_name}

## Stage 1 — Distributed data processing with Apache Spark

We run the ETL as a real **Apache Spark** job on Kubernetes, submitted through
the **Kubeflow Spark Operator**. From the notebook we create a `SparkApplication`
custom resource; the operator spins up **1 Spark driver + 4 executor pods**.
Spark builds a distributed DataFrame of the 60k Fashion-MNIST images and
normalizes, one-hot encodes, and augments them **in parallel across the
executors**, writing one compressed `.npz` shard per partition to the shared
volume.

This is the canonical big-data pattern — and from the notebook's perspective
it's just another Kubernetes resource. **This notebook pod stays idle** while
the Spark cluster does the work.


In [ ]:
from jobs import pipeline

# Submit the Spark ETL job (1 driver + 4 executors) and wait for COMPLETED.
data_job = pipeline.run_data_processing(num_executors=4, num_shards=4, wait=True)


In [ ]:
# Inspect the Spark driver logs and the sharded output on the GCS bucket.
pipeline.print_spark_logs()

print("--- shards ---")
from google.cloud import storage
client = storage.Client()
bucket = client.bucket(bucket_name)
blobs = bucket.list_blobs(prefix="processed/train/")
for blob in blobs:
    print(f"gs://{bucket_name}/{blob.name} ({blob.size} bytes)")

print("--- marker ---")
success_blob = bucket.blob("processed/_SUCCESS")
if success_blob.exists():
    print(success_blob.download_as_text())
else:
    print("_SUCCESS marker not found")

## ⏸️ Pause & Resume — the ML engineer's pain point

Here's the real-world scenario. Stage 2 (TPU training) can run for a long time.
An ML engineer doesn't want to keep a laptop tethered and a notebook pod burning
resources while they wait — but they *also* don't want to lose their in-memory
state (loaded variables, job handles, plots) and re-run everything from scratch.

**Kubeflow Workspaces Snapshot & Restore solves exactly this.** Below we stash
some live state into a variable, then you'll pause the workspace from the
Dashboard. GKE PodSnapshot checkpoints the *entire pod* — kernel memory
included — to GCS. Later you restore it and this variable is still here.

Run the next cell to create some state worth preserving, then follow the
Pause/Resume steps in the markdown that follows.


In [ ]:
import datetime, random

# Live, in-memory state that would normally be lost if the pod were deleted.
snapshot_secret = random.randint(10_000, 99_999)
checkpoint_note = {
    "created_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "data_job_id": data_job,
    "secret_token": snapshot_secret,
    "stage_reached": "data_processing_complete",
}
print("In-memory state to preserve across pause/resume:")
print(checkpoint_note)
print(f"\n>>> Remember this number: {snapshot_secret} <<<")


### 👉 Now pause & resume this workspace

**Option A — from the Dashboard (recommended for the demo):**
1. Open the Workspaces Dashboard: `https://<INGRESS_IP>/workspaces/`
2. Find this workspace (`ml-demo-notebook`) → click **Pause**.
   - GKE PodSnapshot checkpoints the pod (kernel memory + this variable) to GCS.
   - The pod is deleted; you stop paying for the compute.
3. Wait a moment, then click **Resume / Start**.
   - The pod is restored *from the checkpoint*, not restarted cold.
4. Reconnect to this notebook. **Do not re-run the cells above.**

**Option B — from the CLI** (run in a terminal *outside* this pod):
```bash
# Pause (snapshot)
kubectl patch workspace ml-demo-notebook -n default --type merge -p '{"spec":{"paused":true}}'
# ...wait for the snapshot to complete, then resume (restore)
kubectl patch workspace ml-demo-notebook -n default --type merge -p '{"spec":{"paused":false}}'
```

Once the workspace is back, run the cell below. If `snapshot_secret` is still
defined with the **same number** you memorized, the kernel state survived the
pause/resume cycle — no re-execution required.


In [ ]:
# Run this AFTER resuming the workspace — no other cells re-run.
try:
    print("✅ Kernel state survived pause/resume!")
    print(f"   snapshot_secret is still: {snapshot_secret}")
    print(f"   checkpoint_note: {checkpoint_note}")
except NameError:
    print("❌ snapshot_secret is undefined — state was NOT restored.")
    print("   (Did you re-run the cells from the top instead of resuming?)")


## Stage 2 — Distributed model training on TPU

Now the compute-heavy stage. We submit a `TrainJob` that lands on a **multi-host
Cloud TPU v5e slice (2 hosts × 4 cores = 8 TPU cores)**. JAX brings up its
distributed runtime across the hosts, `pmap` parallelizes across cores, and
`jax.lax.pmean` averages gradients across *all 8 cores on both hosts* — true
data-parallel training.

The job reads the preprocessed shards from `/data/processed` and writes the
trained model + metrics back to `/data/model`.

> **TPU quota note:** if you use the `tpu-job-ccc` placeholder to hold TPU
> nodes, suspend it first (see the README) so this TrainJob can claim the slice.


In [ ]:
# Optional: release TPU nodes held by the reservation placeholder job.
!kubectl delete job tpu-job-ccc -n default --ignore-not-found

# Submit the multi-host TPU training job and wait for completion.
train_job = pipeline.run_training(
    num_hosts=2,
    epochs=5,
    global_batch_size=1024,
    wait=True,
)


In [ ]:
# Show training logs and the metrics written to GCS.
pipeline.print_logs(train_job)

import json
from google.cloud import storage
client = storage.Client()
bucket = client.bucket(bucket_name)
blob = bucket.blob("model/metrics.json")
metrics = json.loads(blob.download_as_text())
print("\n=== Final training metrics ===")
print(json.dumps(metrics, indent=2))

## Stage 3 — Inference / serving

The model is trained; now we serve it. We deploy a **2-replica CPU Deployment**
(no TPU needed for this small model) that reads `params.npz` straight off the
shared volume and exposes a `/predict` HTTP endpoint via a `Service`.

The serving code is injected into a stock `python:3.11-slim` image via a
ConfigMap — no custom image build. This shows the same cluster serving traffic
right next to where it trained the model.

Deploy the inference service by running `scripts/apply_inference.sh` from a
terminal (it injects `serve.py` into the ConfigMap and applies the manifest), or
run the cell below which does the same thing from the notebook.


In [ ]:
# Inject serve.py into the ConfigMap, then apply the Deployment + Service.
serve_path = os.path.join(DEMO_DIR, "jobs", "serve.py")

# Build the ConfigMap from serve.py and apply it (idempotent).
cm = subprocess.run(
    ["kubectl", "create", "configmap", "ml-demo-serve-code",
     f"--from-file=serve.py={serve_path}",
     "-n", "default", "--dry-run=client", "-o", "yaml"],
    check=True, capture_output=True, text=True,
).stdout
subprocess.run(["kubectl", "apply", "-f", "-"], input=cm, text=True, check=True)

# Read inference-service.yaml and substitute bucket name
with open(os.path.join(DEMO_DIR, "manifests", "inference-service.yaml")) as f:
    manifest = f.read()
manifest = manifest.replace("BUCKET_NAME_PLACEHOLDER", bucket_name)

# Apply the Deployment + Service. The placeholder ConfigMap embedded in the file
# is harmlessly re-created; the real serve.py above takes precedence on rollout.
subprocess.run(["kubectl", "apply", "-f", "-"], input=manifest, text=True, check=True)

# Re-apply the real ConfigMap (the manifest's placeholder may have overwritten it).
subprocess.run(["kubectl", "apply", "-f", "-"], input=cm, text=True, check=True)
subprocess.run(["kubectl", "rollout", "restart", "deployment/fashion-mnist-inference", "-n", "default"], check=False)

!kubectl rollout status deployment/fashion-mnist-inference -n default --timeout=180s

In [ ]:
# Send real test images to the inference Service and check the predictions.
import os
import json
import numpy as np
import urllib.request
from google.cloud import storage

client = storage.Client()
bucket = client.bucket(bucket_name)
tmp_test = "/tmp/test.npz"
bucket.blob("processed/test/test.npz").download_to_filename(tmp_test)
test = np.load(tmp_test)

sample_idx = [0, 1, 2, 3, 4]
instances = test["images"][sample_idx].tolist()
true_labels = test["labels"][sample_idx].argmax(axis=1).tolist()
os.remove(tmp_test)

CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

req = urllib.request.Request(
    "http://fashion-mnist-inference.default.svc.cluster.local/predict",
    data=json.dumps({"instances": instances}).encode(),
    headers={"Content-Type": "application/json"},
)
resp = json.loads(urllib.request.urlopen(req, timeout=30).read())

print("Predicted vs. true:")
for i, (pred, lbl) in enumerate(zip(resp["labels"], true_labels)):
    mark = "✅" if resp["predictions"][i] == lbl else "❌"
    print(f"  {mark} sample {sample_idx[i]}: predicted={pred:12s}  true={CLASS_NAMES[lbl]}")

## Recap — what just happened

From a single 0.1-CPU Jupyter Workspace, we orchestrated a complete ML lifecycle
across heterogeneous cluster hardware:

- **Stage 1** ran an Apache Spark ETL job (1 driver + 4 executors) to preprocess
  the dataset in parallel — submitted as a `SparkApplication`.
- **Stage 2** trained a model across 8 Cloud TPU cores on 2 hosts (data-parallel
  training with real cross-host gradient all-reduce).
- **Stage 3** served the model from a 2-replica CPU Deployment.
- All three stages exchanged data through one shared GCS bucket.
- We **paused and resumed** the notebook mid-workflow, keeping kernel state
  intact — the long-running orchestration didn't cost us idle compute.

This is the vision: **Kubeflow Notebooks as the gateway to Kubernetes.** The
notebook is where the human works; Kubernetes is the engine that runs the work.

### Cleanup

Run `scripts/cleanup_demo.sh` from a terminal to remove all demo resources
(Spark job, TrainJobs, inference Deployment/Service, ConfigMaps, and the demo
workspace). The cell below removes just the inference service and leaves data/model
intact.

In [ ]:
# Optional: tear down the inference service and SparkConnect session.
!kubectl delete deployment fashion-mnist-inference -n default --ignore-not-found
!kubectl delete service fashion-mnist-inference -n default --ignore-not-found
!kubectl delete sparkconnect fashion-mnist-etl -n default --ignore-not-found
# Re-apply the TPU reservation placeholder to hold the slice, if you use it.
# !kubectl apply -f ../tpu-job-ccc.yaml